# ML Model Validation Framework

An interactive, widget-driven notebook for end-to-end machine learning model validation.  
**Run all cells top to bottom** — each step builds on the previous one.

| Step | Method | What it does |
|------|--------|--------------|
| 1 | `data_loader()` | Choose a built-in sample dataset **or** upload your own CSV |
| 2 | `data_summary()` | Distributions, correlation heatmap, missing-value analysis |
| 3 | `data_info()` | Schema: column names, types, record counts |
| 4 | `data_preprocess()` | Interactive imputation and normalization |
| 5 | `model_prepare()` | Set target variable, task type (classification / regression), train-test split |
| 6 | `feature_select()` | Pearson & Spearman correlation, permutation feature importance |
| 7 | `model_training()` | Compare Logistic Regression, Random Forest, GBM, LightGBM, MLP |
| 8 | `model_register()` | Register the best model(s) for downstream analysis |
| 9 | `model_explain()` | Global & local explainability — SHAP, LIME, PDP |
| 10 | `model_diagnose()` | Accuracy, calibration, overfit, weak spots, resiliency, robustness, fairness |

**Built-in datasets:** Breast Cancer · Wine · Iris · Diabetes · California Housing · GMSC Credit Risk · Taiwan Credit Default  

> **Run on Google Colab** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/minw0607/ml_validation_framework/blob/main/notebooks/ML_Model_Validation.ipynb)

In [ ]:
# ── Setup  (re-run this cell any time you restart the Colab runtime) ──────
import os, sys, subprocess, importlib

REPO        = 'ml_validation_framework'
GITHUB_USER = 'minw0607'

try:
    # ── Colab-specific setup ──────────────────────────────────────────────
    from google.colab import output as _colab_output
    IN_COLAB = True

    # 1. Install dependencies via pip (no restart prompt, safe to re-run)
    print("Installing packages…")
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'eli5', 'shap', 'lime', 'lightgbm', 'xgboost', 'graphviz'],
        check=True
    )
    print("✔ Packages ready.")

    # 2. Enable ipywidgets rendering — must happen AFTER any pip activity
    _colab_output.enable_custom_widget_manager()
    print("✔ Widget manager enabled.")

    # 3. Clone repo on first run; pull latest on every subsequent run
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git'],
            check=True, capture_output=True
        )
        print(f"✔ Cloned {REPO}.")
    else:
        r = subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'],
                           capture_output=True, text=True)
        print(f"✔ Repo: {r.stdout.strip() or r.stderr.strip()}")

    # 4. Add src/ to Python path; reload module if already imported
    src_path = f'{REPO}/src'
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if 'validation' in sys.modules:
        importlib.reload(sys.modules['validation'])

    DATA_DIR = f'{REPO}/data'

except ImportError:
    # ── Local Jupyter fallback ────────────────────────────────────────────
    IN_COLAB = False
    src_path = os.path.join('..', 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    DATA_DIR = os.path.join('..', 'data')

# 5. Import the framework
from validation import ValidationFramework

print(f"✔ Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"✔ Data dir    : {os.path.abspath(DATA_DIR)}")
print("\nSetup complete — proceed to Step 1 below.")


## Step 1 — Load Data

Initialise the validation object and load your dataset. You can choose from several **built-in sample datasets** (credit risk, breast cancer, housing, etc.) or upload your own **CSV file**.

| Control | What it does |
|---------|-------------|
| Dataset dropdown | Pick a built-in dataset — task type & target are pre-filled |
| Upload CSV | Select any CSV from your machine; the framework infers delimiter & encoding |

> **Tip:** If uploading your own data, make sure the file has a header row. Unsupported column names (spaces, special chars) are auto-sanitised.

In [ ]:
val = ValidationFramework(data_dir=DATA_DIR)
val.data_loader()

## Step 2 — Data Summary

Get a quick statistical overview of the loaded dataset: row/column counts, per-column data types, missing-value rates, and descriptive statistics (mean, std, quartiles).

Use the **tabs** to switch between the numeric summary and the categorical summary.

> **Tip:** High missing-value rates (> 20 %) on key columns often need explicit handling in the Preprocessing step.

In [ ]:
val.data_summary()

## Step 3 — Data Quality & Correlation

Deep-dive into individual feature distributions and pairwise relationships:

- **Distribution** — histogram + KDE for any selected numeric column
- **Scatter** — pairwise scatter plot to spot non-linear relationships
- **Pearson / Spearman heatmaps** — measure linear and rank-based correlation

> **Tip:** Strong predictor–predictor correlations (|r| > 0.8) signal multicollinearity — consider dropping one of the pair in Feature Selection.

In [ ]:
val.data_info()

## Step 4 — Preprocessing

Clean and transform the raw data before modelling:

- **Missing values** — choose to drop rows, mean/median/mode-impute, or leave as-is
- **Encoding** — one-hot or ordinal encoding for categorical columns
- **Scaling** — optional standard or min-max scaling for numeric columns

A live preview of the transformed dataset updates after each change.

> **Tip:** Tree-based models (RF, GBM, LightGBM) are scale-invariant — scaling is mainly needed for linear models and neural networks.

In [ ]:
val.data_preprocess()

## Step 5 — Model Setup

Define the modelling problem and configure the experiment:

| Setting | Description |
|---------|------------|
| **Target variable** | The column the model should predict |
| **Task type** | Classification or Regression (auto-detected for built-in datasets) |
| **Train / test split** | Fraction of data held out for evaluation (default 0.3) |
| **Random seed** | Fix for reproducible splits and model initialisation |

> **Tip:** If you loaded a built-in dataset the task type and target variable are already pre-filled — just confirm and click **Confirm**.

In [ ]:
val.model_prepare()

## Step 6 — Feature Selection

Identify and retain the most predictive features using two complementary methods:

- **Correlation filtering** — Pearson & Spearman correlation with the target; remove features below a threshold
- **Permutation Importance** — fit a Random Forest (or other model), shuffle each feature and measure the accuracy drop

Adjust the **importance threshold slider** to filter low-signal features, then click **Remove Features**. Use **Revert** to restore all features.

> **Tip:** Start with a permissive threshold (e.g. 0.01) and tighten gradually. Removing too many features at once can hurt model performance unexpectedly.

In [ ]:
val.feature_select()

## Step 7 — Model Training & Evaluation

Train one or more candidate models and compare their performance:

**Algorithms available:** Logistic Regression · Random Forest · Gradient Boosting · LightGBM · MLP Neural Network

**Metrics shown:**
- *Classification* — Accuracy, AUC, F1, Precision, Recall + Confusion Matrix + ROC curve
- *Regression* — R², MAE, RMSE

> **Tip:** For imbalanced datasets prefer **AUC** as the primary comparison metric rather than Accuracy.

In [ ]:
val.model_training()

## Step 8 — Model Registration

Save the trained model(s) and log their metadata for reproducibility:

- Model artifact serialised to disk (`.pkl`)
- Metadata recorded: algorithm, hyperparameters, evaluation metrics, timestamp
- Registered models can be reloaded for inference or further validation

> **Tip:** Give your model a descriptive name (e.g. `rf_credit_2026`) so it is easy to identify when comparing multiple experiments.

In [ ]:
val.model_register()

## Step 9 — Model Explainability

Understand *why* the model makes its predictions using three techniques:

| Method | Scope | Best used for |
|--------|-------|---------------|
| **SHAP** | Global & local | Feature attribution across all predictions |
| **PDP** | Global | Marginal effect of one or two features (one-way & two-way) |
| **LIME** | Local | Surrogate explanation for a single prediction |

> **Tip:** Use SHAP beeswarm for global insight. Switch to LIME or SHAP local plots to explain individual high-risk predictions. PDP works best when features are not strongly correlated with each other.

In [ ]:
val.model_explain()

## Step 10 — Model Diagnostics

Stress-test and audit the trained model across multiple diagnostic dimensions. Each tab below runs a targeted test:

| Tab | What it checks |
|-----|----------------|
| **Accuracy** | Segment-level performance — does the model perform equally well across groups? |
| **Stability** | Performance drift across time slices or data subsets |
| **Conformal Prediction** | Coverage guarantee — how often do prediction sets contain the true label? |
| **Overfitting** | Train/test metric gap · learning curves · feature perturbation sensitivity |
| **Weak Spot** | One-way & two-way heatmaps highlighting underperforming feature segments |
| **Robustness** | Sensitivity to input noise and feature perturbations |

> **Tip:** Run all tabs before signing off on a model. The **Overfitting** and **Weak Spot** tabs often surface issues that aggregate metrics miss entirely.

In [ ]:
val.model_diagnose()